In [1]:
import sys
import os
import time
from lightkurve import search_targetpixelfile
from lightkurve import TessTargetPixelFile
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from keras.models import load_model
from keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Conv1D, MaxPooling1D, Flatten
from wotan import slide_clip
from wotan import transit_mask, flatten
from astropy.stats import sigma_clip
from astropy import units as u
import csv
import shutil
from scipy.interpolate import interp1d
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import make_forecasting_frame
from tsfresh.utilities.dataframe_functions import impute
from statsmodels.tsa.seasonal import seasonal_decompose
from multiprocessing import Pool
import multiprocessing
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.signal import find_peaks
from astropy.timeseries import BoxLeastSquares
from scipy.interpolate import interp1d
from keras.models import Model
from keras.layers import Input, Dense, Conv1D, Flatten, Concatenate, Dropout
from keras.optimizers import Adam

/Users/nasvinnabeel/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
%matplotlib inline

In [3]:
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

Ensemble Model

In [4]:
value_df = 10000
cwd = os.getcwd()
dirname = cwd[len(cwd)-len("Satellite Datasets"):len(cwd)]
if(dirname != 'Satellite Datasets'):
    os.chdir('./Satellite Datasets')
star_check = pd.read_csv("updated_database_exoplanet.csv")
star_check = star_check.drop(['tid'],axis=1)
star_check_y = star_check[['confirmed_planet']]
star_check = star_check.reset_index().drop('index',axis=1)
star_check = star_check.apply(lambda row: row.fillna(0), axis=1)

In [5]:
os.chdir('../')

In [6]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
# star_check[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']] = scaler.fit_transform(star_check[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']])
star_check_x = star_check.drop('confirmed_planet', axis=1)
star_check_scaled = scaler.fit_transform(star_check_x)
star_check_scaled_df = pd.DataFrame(star_check_scaled, columns=star_check_x.columns)
star_check_scaled_df['confirmed_planet'] = star_check_y.values

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(star_check.drop('confirmed_planet',axis=1),star_check[['confirmed_planet']], test_size=0.1, random_state=42)

In [8]:
X_train_flux = X_train.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_train_params = X_train[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]
X_test_flux = X_test.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_test_params = X_test[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]

In [9]:
input_flux = Input(shape=(10000,1))
x = Conv1D(filters=32, kernel_size=5, activation='relu')(input_flux)
x = MaxPooling1D(pool_size=2)(x)
x = Flatten()(x)
model_flux = Model(inputs=input_flux, outputs=x)

input_params = Input(shape=(11,))
y = Dense(128, activation='relu')(input_params)
model_params = Model(inputs=input_params, outputs=y)

combined = Concatenate()([model_flux.output, model_params.output])
z = Dense(128, activation='relu')(combined)
final_output = Dense(1, activation='sigmoid')(z)

model = Model(inputs=[model_flux.input, model_params.input], outputs=final_output)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# history = model.fit([X_train_flux, X_train_params], y_train, epochs=15, batch_size=32, validation_split=0.2)


2024-03-26 19:24:27.494239: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2024-03-26 19:24:27.494338: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2024-03-26 19:24:27.494348: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2024-03-26 19:24:27.494376: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-03-26 19:24:27.494389: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [10]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  # Metric to monitor
    patience=10,         # Number of epochs with no improvement after which training will be stopped
    mode='min'          # The training will stop when the quantity monitored has stopped decreasing
)

class MyThresholdCallback(tf.keras.callbacks.Callback):
    def __init__(self, threshold):
        super(MyThresholdCallback, self).__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs["val_accuracy"]
        acc = logs['accuracy']
        if val_acc >= self.threshold and acc >= self.threshold:
            self.model.stop_training = True

my_callback = MyThresholdCallback(threshold=0.9)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# history = model.fit([X_train_flux, X_train_params], y_train, epochs=100, batch_size=32, validation_split=0.2, callbacks=[my_callback, early_stopping])

In [11]:
# model.load_weights('model_weights_88_accuracy.weights.h5')
model.load_weights('model_weights.h5')

In [17]:
model.evaluate([X_test_flux,X_test_params],y_test)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8894 - loss: 0.3394


[0.3036459684371948, 0.9004328846931458]

In [13]:
# model.save_weights('model_weights_88_accuracy.weights.h5')